# Task 5: LoRA Fine-Tuning

Trains the Baroque and Romantic adapters using the preserved `anticipatory-sequence-v9` caches. The base model (stanford-crfm/music-medium-800k) was pretrained on **Lakh MIDI**, not MAESTRO.

Restore the saved experiment configuration and pass the real-cache CUDA smoke test before running either full training command. Cache-only mode does not require the MAESTRO manifest or original MIDI files. Training retains periodic and final checkpoints; it does not select a best checkpoint by validation objective.

In [ ]:
# Cell 1: Mount Drive and clone the active branch
from google.colab import drive
drive.mount('/content/drive')

import subprocess
import os
repo_dir = '/content/thesis'
repo_url = 'https://github.com/faraz-irshad/music-style-transformer.git'
if os.path.isdir(repo_dir):
    subprocess.run(['git', '-C', repo_dir, 'fetch', 'origin', 'fixes-and-ui'], check=True)
    subprocess.run(['git', '-C', repo_dir, 'checkout', '-B', 'fixes-and-ui', 'origin/fixes-and-ui'], check=True)
else:
    subprocess.run(['git', 'clone', '-b', 'fixes-and-ui', '--single-branch', repo_url, repo_dir], check=True)
os.chdir(repo_dir)
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print(f'Using fixes-and-ui commit {commit_hash}')

In [ ]:
# Cell 2: Install dependencies
exec(open('colab/setup_cell.py').read())

In [ ]:
# Cell 3: Restore the saved experiment configuration exactly; never merge defaults
from pathlib import Path
import yaml
SAVED_CONFIG = Path('/content/drive/MyDrive/thesis_results/alignment_v1_data/config.alignment_v1.yaml')
if not SAVED_CONFIG.is_file():
    raise FileNotFoundError(f'Saved alignment_v1 config is missing: {SAVED_CONFIG}')
with SAVED_CONFIG.open() as f:
    cfg = yaml.safe_load(f)
required = {'model', 'lora', 'training'}
missing = required - set(cfg or {})
if missing:
    raise ValueError(f'Saved config is missing required sections: {sorted(missing)}')
if cfg['training'].get('invalid_window_policy') != 'fail-on-any-window-error-empty-or-all-masked-v1':
    raise ValueError('Saved config does not use the required fail-closed window policy')
print(f'Restored exact saved config: {SAVED_CONFIG}')
print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
# Cell 4: Real-cache CUDA smoke test (one update + both validation forwards)
import hashlib, json, shutil, subprocess, sys
from pathlib import Path
import yaml
from model.training.train_lora import discover_cache_selection

saved_config = Path('/content/drive/MyDrive/thesis_results/alignment_v1_data/config.alignment_v1.yaml')
if not saved_config.is_file():
    raise FileNotFoundError(f'Saved alignment_v1 config is missing: {saved_config}')
with saved_config.open() as f:
    smoke_cfg = yaml.safe_load(f)
experiment_root = Path(smoke_cfg['training'].get('experiment_dir') or
                        Path(smoke_cfg['training']['drive_dir']) / 'experiments' /
                        smoke_cfg['training']['experiment_name'])
cache_dir = Path('/content/drive/MyDrive/thesis_adapters/experiments/neural_alignment_v1/token_cache')
if (experiment_root / 'token_cache').resolve() != cache_dir.resolve():
    raise ValueError(f'Saved config resolves to a different cache directory: {experiment_root / "token_cache"}')
selection = discover_cache_selection(cache_dir, smoke_cfg)
selection_path = Path('/content/drive/MyDrive/thesis_results/alignment_v1_data/cache_selection.v1.json')
selection_text = json.dumps(selection, indent=2, sort_keys=True) + '\n'
if selection_path.exists():
    if selection_path.read_text() != selection_text:
        raise FileExistsError(f'Refusing to replace different cache selection: {selection_path}')
else:
    selection_path.write_text(selection_text)

# Preserve any surviving exclusion/manifest evidence without replacing Drive copies.
evidence_source = Path('/content/thesis_data_alignment_v1')
evidence_names = ('manifest', 'exclusion', 'reject', 'invalid', 'diagnostic')
evidence_files = ([p for p in evidence_source.iterdir() if p.is_file() and
                   any(word in p.name.lower() for word in evidence_names)]
                  if evidence_source.is_dir() else [])
evidence_destination = Path('/content/drive/MyDrive/thesis_results/alignment_v1_data/provenance')
evidence_destination.mkdir(parents=True, exist_ok=True)
evidence_record = {'source': str(evidence_source), 'files': []}
if evidence_files:
    evidence_destination.mkdir(parents=True, exist_ok=True)
    for source in evidence_files:
        target = evidence_destination / source.name
        source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
        if target.exists():
            if hashlib.sha256(target.read_bytes()).hexdigest() != source_hash:
                raise FileExistsError(f'Refusing to overwrite differing evidence: {target}')
        else:
            shutil.copy2(source, target)
        evidence_record['files'].append({'name': source.name, 'sha256': source_hash})
    evidence_record['status'] = 'preserved'
    print(f'Preserved {len(evidence_files)} surviving exclusion/manifest evidence file(s) on Drive.')
else:
    evidence_record['status'] = 'provenance_gap'
    evidence_record['reason'] = 'No local exclusion log or manifest backup was found.'
    print('PROVENANCE GAP: no local exclusion log or manifest backup found; the two excluded Romantic windows cannot be independently recovered from these caches.')
evidence_record_path = evidence_destination / 'exclusion_provenance.json'
evidence_record_text = json.dumps(evidence_record, indent=2, sort_keys=True) + '\n'
if evidence_record_path.exists():
    if evidence_record_path.read_text() != evidence_record_text:
        raise FileExistsError(f'Refusing to overwrite differing provenance record: {evidence_record_path}')
else:
    evidence_record_path.write_text(evidence_record_text)

smoke_command = [sys.executable, '-u', 'train_lora.py', '--config', str(saved_config),
                 '--style', 'baroque', '--cache-only', '--cache-selection',
                 str(selection_path), '--smoke-test']
print('Running:', ' '.join(smoke_command))
subprocess.run(smoke_command, check=True)
NEURAL_ALIGNMENT_V1_SMOKE_PASSED = True

In [ ]:
# Cell 5: Full training — only after the real-cache smoke-test cell passes
import subprocess, sys
if globals().get('NEURAL_ALIGNMENT_V1_SMOKE_PASSED') is not True:
    raise RuntimeError('Run and pass Cell 4 before full training.')
saved_config = '/content/drive/MyDrive/thesis_results/alignment_v1_data/config.alignment_v1.yaml'
selection_path = '/content/drive/MyDrive/thesis_results/alignment_v1_data/cache_selection.v1.json'
for style in ('baroque', 'romantic'):
    command = [sys.executable, '-u', 'train_lora.py', '--config', saved_config,
               '--style', style, '--cache-only', '--cache-selection', selection_path]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)

In [ ]:
# Cell 7: Plot training curves
import json, matplotlib.pyplot as plt
from pathlib import Path

drive_dir = cfg['training']['drive_dir']
experiment_root = Path(cfg['training'].get('experiment_dir') or
                       Path(drive_dir) / 'experiments' /
                       cfg['training'].get('experiment_name', 'neural_alignment_v1'))
plot_path = experiment_root / 'training_curves.png'
if plot_path.exists():
    raise FileExistsError(f'Refusing to overwrite {plot_path}')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, style in zip(axes, ('baroque', 'romantic')):
    log_path = experiment_root / f'adapter_{style}' / 'train_log.jsonl'
    if not log_path.exists():
        ax.set_title(f'{style} — no log')
        continue
    steps, train_losses = [], []
    unconditional_losses, conditioned_losses, val_steps = [], [], []
    with open(log_path) as f:
        for line in f:
            entry = json.loads(line)
            steps.append(entry['step'])
            train_losses.append(entry['train_loss'])
            if 'val_unconditional_piano_loss' in entry:
                val_steps.append(entry['step'])
                unconditional_losses.append(entry['val_unconditional_piano_loss'])
                conditioned_losses.append(entry['val_conditioned_accompaniment_loss'])
    ax.plot(steps, train_losses, label='train')
    if unconditional_losses:
        ax.plot(val_steps, unconditional_losses, label='val unconditional piano', marker='o')
        ax.plot(val_steps, conditioned_losses, label='val conditioned accompaniment', marker='o')
    ax.set_title(f'{style} adapter')
    ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.legend()

plt.tight_layout()
plt.savefig(plot_path, dpi=120)
plt.show()